In [0]:
CATALOG = "energy_puls"
BRONZE = f"{CATALOG}.bronze"
SILVER = f"{CATALOG}.silver"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")

In [0]:
display(spark.sql(f"""
  SELECT date_heure, libelle_region, COUNT(*) AS times_received
  FROM {BRONZE}.eco2mix_api_raw
  GROUP BY 1,2
  HAVING COUNT(*) > 1
  ORDER BY times_received DESC
  LIMIT 10
"""))

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

w = Window.partitionBy("date_heure", "libelle_region") \
          .orderBy(F.col("pull_timestamp").desc())

silver = (spark.table(f"{BRONZE}.eco2mix_api_raw")
    .withColumn("_rn", F.row_number().over(w))
    .filter("_rn = 1")         
    .drop("_rn"))

silver.write.mode("overwrite").saveAsTable(f"{SILVER}.eco2mix_clean")

print("Bronze:", spark.table(f"{BRONZE}.eco2mix_api_raw").count())
print("Silver:", spark.table(f"{SILVER}.eco2mix_clean").count())